# PROJECT 18: FINAL DEMO KHOA HỌC
## Low-Light Image Enhancement and Downstream Recognition
**Mô hình sử dụng:**
- **Zero-DCE:** `Results/weights/zerodce_best.pth` (10 Epochs, 79,240 tham số)
- **YOLOv8n Dark Baseline:** `Results/weights/yolov8n_dark_best.pt` (Huấn luyện 18-19 / 40 Epochs trên Kaggle GPU, Best mAP@50: 25.6%)

In [ ]:
import sys
sys.path.append('..')
from pathlib import Path
import time
import cv2
import torch
import numpy as np
import matplotlib.pyplot as plt
from ultralytics import YOLO
from src.model_zerodce import DCENet
from src.visualize import draw_bounding_boxes

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"💻 Thiết bị tính toán: {device.type.upper()}")

### 1. Nạp Trọng Số Huấn Luyện & Đọc Metadata Checkpoint

In [ ]:
dce_weight = Path('../Results/weights/zerodce_best.pth')
yolo_weight = Path('../Results/weights/yolov8n_dark_best.pt')

# Nạp Zero-DCE
model_dce = DCENet().to(device)
model_dce.load_state_dict(torch.load(dce_weight, map_location=device))
model_dce.eval()
print(f"✅ Đã nạp Zero-DCE ({sum(p.numel() for p in model_dce.parameters()):,} tham số)")

# Đọc Checkpoint YOLOv8
ckpt = torch.load(yolo_weight, map_location='cpu', weights_only=False)
epoch_idx = int(ckpt.get('epoch', 17)) + 1
best_fit = float(ckpt.get('best_fitness', 0.255))
print(f"✅ Đã nạp YOLOv8n từ Kaggle Checkpoint:")
print(f"   • Số Epochs huấn luyện: {epoch_idx} / 40 Epochs")
print(f"   • Best Fitness (mAP@50): {best_fit*100:.1f}%")

model_yolo = YOLO(str(yolo_weight))

### 2. Thực Thi Pipeline Đối Chứng 4 Khung Hình (Side-by-Side)

In [ ]:
# Chọn ảnh mẫu tiêu biểu từ tập test ExDark (Xe buýt, người đi bộ, xe đạp)
img_path = Path('../Dataset/exdark_yolo_dark/test/images/2015_02021_jpg.rf.ee8738d037d08f6f45e91d1ebf55c34a.jpg')
raw_bgr = cv2.imread(str(img_path))
raw_rgb = cv2.cvtColor(raw_bgr, cv2.COLOR_BGR2RGB)

# Giai đoạn 1: Làm sáng ảnh qua Zero-DCE
t0 = time.perf_counter()
t_in = torch.from_numpy(raw_rgb).float().permute(2, 0, 1).unsqueeze(0) / 255.0
with torch.no_grad():
    enh_t, _ = model_dce(t_in.to(device))
enh_np = (enh_t.squeeze(0).permute(1, 2, 0).cpu().numpy() * 255.0).clip(0, 255).astype(np.uint8)
time_dce = (time.perf_counter() - t0) * 1000.0

# Giai đoạn 2A: Nhận diện trên ảnh tối gốc
t0 = time.perf_counter()
res_dark = model_yolo.predict(raw_rgb, conf=0.20, verbose=False)
time_yolo_dark = (time.perf_counter() - t0) * 1000.0
boxes_dark = res_dark[0].boxes
det_dark = draw_bounding_boxes(raw_rgb.copy(), boxes_dark, model_yolo.names)

# Giai đoạn 2B: Nhận diện trên ảnh đã làm sáng
t0 = time.perf_counter()
res_enh = model_yolo.predict(enh_np, conf=0.20, verbose=False)
time_yolo_enh = (time.perf_counter() - t0) * 1000.0
boxes_enh = res_enh[0].boxes
det_enh = draw_bounding_boxes(enh_np.copy(), boxes_enh, model_yolo.names)

print(f"⏱️ Thời gian xử lý: Zero-DCE={time_dce:.1f}ms | YOLOv8={time_yolo_enh:.1f}ms | Tổng={(time_dce+time_yolo_enh):.1f}ms")
print(f"🎯 Số vật thể phát hiện trên ảnh tối gốc: {len(boxes_dark)} vật thể")
print(f"🎯 Số vật thể phát hiện trên ảnh sáng Zero-DCE: {len(boxes_enh)} vật thể")

### 3. Trực Quan Hóa 4 Khung Hình Khoa Học

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

axes[0, 0].imshow(raw_rgb)
axes[0, 0].set_title("1. Ảnh Tối Gốc (Input Dark Scene)", fontsize=13, fontweight="bold")
axes[0, 0].axis("off")

axes[0, 1].imshow(enh_np)
axes[0, 1].set_title(f"2. Đã Làm Sáng (Zero-DCE) [{time_dce:.0f}ms]", fontsize=13, fontweight="bold", color="#0284c7")
axes[0, 1].axis("off")

axes[1, 0].imshow(det_dark)
axes[1, 0].set_title(f"3. YOLOv8 Trên Ảnh Tối ({len(boxes_dark)} vật thể) [{time_yolo_dark:.0f}ms]", fontsize=13, fontweight="bold")
axes[1, 0].axis("off")

axes[1, 1].imshow(det_enh)
axes[1, 1].set_title(f"4. YOLOv8 Trên Ảnh Sáng ({len(boxes_enh)} vật thể) [{time_yolo_enh:.0f}ms]", fontsize=13, fontweight="bold", color="#16a34a")
axes[1, 1].axis("off")

plt.suptitle(f"PROJECT 18: KẾT QUẢ ĐỐI CHỨNG CHECKPOINT KAGGLE ({epoch_idx}/40 EPOCHS - mAP@50: {best_fit*100:.1f}%)", fontsize=15, fontweight="bold", y=0.98)
plt.tight_layout(rect=[0, 0.02, 1, 0.95])
plt.show()